In [ ]:
from dataclasses import dataclass
from datetime import datetime, time
from enum import Enum
from typing import Dict

# Output Models

class TariffStatus(Enum):
    OFF_PEAK = "OFF-PEAK"   # Low priority
    NORMAL = "NORMAL"       # Normal
    PEAK = "PEAK"           # High priority

@dataclass(frozen=True)
class CostContextRecord:
    """Economic assessment."""
    timestamp: datetime
    current_tariff: float
    tariff_status: TariffStatus
    cost_severity_index: int  # 1-10 scale

# Module Implementation

class EnerviaCostContext:
    """Tariff Context Engine."""

    def __init__(self, peak_threshold_multiplier: float = 1.5):
        """Initialize."""
        # Base rate
        self.base_tariff = 6.0
        
        # ToU windows
        self.peak_windows = [
            (time(10, 0), time(14, 0)),  # Day peak
            (time(18, 0), time(22, 0))   # Eve peak
        ]
        self.off_peak_windows = [
            (time(0, 0), time(6, 0))     # Night off-peak
        ]

    def evaluate_cost_context(self, current_time: datetime, current_tariff: float) -> CostContextRecord:
        """Evaluate cost context."""
        
        # Time status
        status = self._classify_time_window(current_time.time())
        
        # Price status
        if current_tariff >= (self.base_tariff * 1.8):
            status = TariffStatus.PEAK
            
        # Calc severity
        severity = self._calculate_severity(status, current_tariff)

        return CostContextRecord(
            timestamp=current_time,
            current_tariff=current_tariff,
            tariff_status=status,
            cost_severity_index=severity
        )

    def _classify_time_window(self, t: time) -> TariffStatus:
        """Classify time window."""
        for start, end in self.peak_windows:
            if start <= t <= end:
                return TariffStatus.PEAK
        
        for start, end in self.off_peak_windows:
            if start <= t <= end:
                return TariffStatus.OFF_PEAK
                
        return TariffStatus.NORMAL

    def _calculate_severity(self, status: TariffStatus, tariff: float) -> int:
        """Calculate severity."""
        base_severity = {
            TariffStatus.OFF_PEAK: 1,
            TariffStatus.NORMAL: 5,
            TariffStatus.PEAK: 8
        }
        
        severity = base_severity[status]
        
        # Price modifier
        if tariff > self.base_tariff * 2.0:
            severity = min(10, severity + 2)
        elif tariff > self.base_tariff * 1.2:
            severity = min(10, severity + 1)
            
        return severity


# engine = EnerviaCostContext()
# cost_report = engine.evaluate_cost_context(datetime.now(), 12.5) 
# print(f"Status: {cost_report.tariff_status}, Severity: {cost_report.cost_severity_index}")